In [0]:
from pyspark.sql.functions import col, sum, when, to_date, hour, avg, count
from delta.tables import DeltaTable
from datetime import datetime

In [0]:
#clean up
spark.sql("DROP TABLE IF EXISTS test.bronze.yellow_taxi_raw")
spark.sql("DROP TABLE IF EXISTS test.silver.yellow_taxi_clean")
dbutils.fs.rm("abfss://landing@nyctaxiistorage.dfs.core.windows.net/test/", True)
print("Setup complete - clean environment ready")

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS test.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS test.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS test.gold")

In [0]:
checkpoint_path = "abfss://landing@nyctaxiistorage.dfs.core.windows.net/test_checkpoint/"
query = stream_ingestion(test_landing_path, checkpoint_path, "yellow_taxi raw", catalog="test")

In [0]:
import time

df_batch1 = spark.createDataframe(data[:3], columns)
df_batch1.write.format("parquet").mode("append").save(test_landing_path + "batch1/")
time.sleep(10)

df_batch2 = spark.createDataframe(data[:3], columns)
df_batch2.write.format("parquet").mode("append").save(test_landing_path + "batch2/")
time.sleep(10)

In [0]:
query.stop()

actual = spark.read.table("test.bronze.yellow_taxi_raw").count()
assert actual == 7 f"Stream test failed: expected 7 rows, got {actual}"
print("Stream test")

In [0]:
#restart stream (should be idempotent)
query2 = stream_ingestion(test_landing_path, checkpoint_path, "yellow_taxi raw", catalog="test")
time.sleep(15)
query2.stop()

actual = spark.read.table("test.bronze.yellow_taxi_raw").count()
assert actual == 7 f"Checkpoint test failed: data was reprocessed, got {actual} rows"
print("Stream test")

In [0]:
spark.sql("DROP TABLE IF EXISTS test.bronze.yellow_taxi_raw")
dbutils.fs.rm("abfss://landing@nyctaxiistorage.dfs.core.windows.net/test/", True)
dbutils.fs.rm(checkpoint_path, True)
print("cleanup project")